In [1]:
pip install pretty_midi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 76.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.4 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=0b64629171c9e67dcc4db521fb9056acd28e1f3cd4a2392ae0cffabf05498b92
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pretty_midi
from tqdm import tqdm
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#paths
csv_path = Path("/content/drive/MyDrive/CSE425 Project/data/maestro-v3.0.0.csv")
input_path = Path("/content/drive/MyDrive/CSE425 Project/data/raw_data")
output_path = Path('/content/drive/MyDrive/CSE425 Project/data/train-val-test')
output_path.mkdir(parents=True, exist_ok=True)
processed_info_path = Path('/content/drive/MyDrive/CSE425 Project/data/processed_info')
processed_info_path.mkdir(parents=True, exist_ok=True)
plots_path = Path('/content/drive/MyDrive/CSE425 Project/output/EDA')
plots_path.mkdir(parents=True, exist_ok=True)

print(f"CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV : {csv_path}")
print(f"Reading from: {input_path}")
print(f"Writing to: {output_path}")
print(f"Will save config/stats to: {processed_info_path}")
midi_count = len(list(input_path.glob('**/*.mid'))) + len(list(input_path.glob('**/*.midi')))
print(f"Found {midi_count} MIDI files")

CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV CSV : /content/drive/MyDrive/CSE425 Project/data/maestro-v3.0.0.csv
Reading from: /content/drive/MyDrive/CSE425 Project/data/raw_data
Writing to: /content/drive/MyDrive/CSE425 Project/data/train-val-test
Will save config/stats to: /content/drive/MyDrive/CSE425 Project/data/processed_info
Found 1276 MIDI files


### FOR TASK 1 AND 2

In [5]:
# --- Configuration ---
FS = 16
WINDOW_LENGTH = 128
MIN_ACTIVITY_THRESHOLD = 0.02
PIANO_MIN = 21
PIANO_MAX = 108

In [6]:
# WE READ CSV
df = pd.read_csv(csv_path)

In [7]:
def extract_windows_from_split(split_name, df):
    split_df = df[df['split'] == split_name]
    all_windows = []

    print(f"\nProcessing {split_name} split ({len(split_df)} files)...")

    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        midi_path = input_path / row['midi_filename']

        if not midi_path.exists():
            continue
        try:
            pm = pretty_midi.PrettyMIDI(str(midi_path))
            pr = pm.get_piano_roll(fs=FS)
            pr = pr[PIANO_MIN:PIANO_MAX+1, :] # Rows 21-108 (88 keys)
            pr = pr.T # Transpose to (T, 88)
            pr_binary = (pr > 0).astype(np.float32)
            num_windows = pr_binary.shape[0] // WINDOW_LENGTH
            for i in range(num_windows):
                window = pr_binary[i*WINDOW_LENGTH : (i+1)*WINDOW_LENGTH, :]
                if np.mean(window) >= MIN_ACTIVITY_THRESHOLD:
                    all_windows.append(window)
        except:
            continue
    if all_windows:
        final_array = np.stack(all_windows, axis=0)
        save_path = output_path / f'piano_roll_{split_name}.npy'
        np.save(save_path, final_array)
        print(f"WE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \nWE GOT THIS \n {len(final_array)} windows to {save_path}")
    else:
        print(f"No windows passed the threshold for {split_name}")


In [8]:
maestro_df = pd.read_csv(csv_path)
for s in ['train', 'validation', 'test']:
    extract_windows_from_split(s, maestro_df)


Processing train split (962 files)...


100%|██████████| 962/962 [16:22<00:00,  1.02s/it]


WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
 62689 windows to /content/drive/MyDrive/CSE425 Project/data/train-val-test/piano_roll_train.npy

Processing validation split (137 files)...


100%|██████████| 137/137 [02:12<00:00,  1.03it/s]


WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
 7876 windows to /content/drive/MyDrive/CSE425 Project/data/train-val-test/piano_roll_validation.npy

Processing test split (177 files)...


100%|██████████| 177/177 [02:33<00:00,  1.16it/s]


WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
WE GOT THIS 
 7792 windows to /content/drive/MyDrive/CSE425 Project/data/train-val-test/piano_roll_test.npy
